# OSM

 - [Wiki Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API#Building_blocks)
 - [Python Overpass API](https://python-overpy.readthedocs.io/en/latest/example.html)
 - [Loading data from OSM with Python and the Overpass API](https://towardsdatascience.com/loading-data-from-openstreetmap-with-python-and-the-overpass-api-513882a27fd0)
 - [PyGIS - Accessing OSM data in python](https://pygis.io/docs/d_access_osm.html)
 - [OSM tags Wiki](https://wiki.openstreetmap.org/wiki/Tags)
 - [OSM and urban data, in book "Geospatial Analysis with Python and R"](https://kodu.ut.ee/~kmoch/geopython2021/L4/osm-urban.html)

### Load

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# %load import.py
#
from IPython.display import IFrame

# 
import numpy as np
import pandas as pd

#
from sentinelsat import SentinelAPI, read_geojson, geojson_to_wkt
from datetime import date

#
import rioxarray
import geopandas as gpd
import rasterio as rio

#
from matplotlib import pyplot
from rasterio.plot import show

#
from sqlalchemy import create_engine # query PostGIS
from sqlalchemy import inspect

#
import osmnx as ox

#
from shapely.geometry import Polygon, box
import shapely.ops as so

#
import json

#
import os
from pathlib import Path
import fnmatch
import glob

#
from osgeo import gdal

#
import random

#
import shutil

#
from tqdm import tqdm
import time

#
import folium
from folium import plugins

## Retrieve tags

### Get BBox from Selected Products

#### Load GeoDF

In [ ]:
sel_prod_gdf = geopandas.read_file("sel_products.geojson")

#### Check GeoDF

In [ ]:
sel_prod_gdf

In [ ]:
type(sel_prod_gdf)

### TAGS definition

#### Buildings

In [ ]:
# List key-value pairs for tags
tags = {'building': True}   

### Convert GeoDataFrame geometry (polygon / multipolygon) into Shapely polygon

#### Use the Products bounding box | too large | <i style="color:magenta">not working</b>

In [ ]:
x,y = sel_prod_gdf.explode(index_parts=False).geometry[0].exterior.coords.xy

In [ ]:
sel_prod_pol = Polygon(list(zip(x, y)))

In [ ]:
sel_prod_pol

Retrieve OSM data

In [ ]:
osm_data = ox.geometries_from_polygon( sel_prod_pol, tags )

Save OSM data

In [ ]:
osm_data.to_file("osm_data/buildings.geojson", driver='GeoJSON')

#### Use the Study Area bounding box | smaller | <i style="color:magenta">not working</b>

In [ ]:
x,y = sa.explode(index_parts=False).geometry[0].exterior.coords.xy

In [ ]:
sel_prod_pol = Polygon(list(zip(x, y)))

In [ ]:
sel_prod_pol

Retrieve OSM data

In [ ]:
osm_data = ox.geometries_from_polygon( sel_prod_pol, tags )

Save OSM data

In [ ]:
osm_data.to_file("osm_data/buildings.geojson", driver='GeoJSON')

### Use PostGIS cities within Products bounding box

#### Retrieve OSM data

In [ ]:
osm_data = ox.geometries_from_place( cities_gdf["nuts_name"][0], tags )

In [ ]:
osm_data.head(3).plot()

In [ ]:
osm_data.head(3)

#### Save OSM data

In [ ]:
fname = "osm_data/buildings__" + cities_gdf["nuts_name"][0] + ".geojson"
osm_data.to_file( fname, driver='GeoJSON' )

#### OSM data :: BATCH retrieve + save

In [ ]:
dir_osmdata = "osm_data/"

In [ ]:
tags

In [ ]:
osm_data = ox.geometries_from_place( 'Moschiano', tags )

In [ ]:
len(osm_data)

In [ ]:
for row, fields in cities_gdf.iterrows():
    Found = False
    for filename in os.listdir( dir_osmdata ):
        if filename.startswith( "buildings__" ):
            if fnmatch.fnmatch( filename, 'buildings__' + fields[0] + '.geojson' ):
                print( "%04d" % row, "%40s" % fields[0], '  [existent]')
                Found = True
    if not Found:
        try:
            print( "%04d" % row, "%40s" % fields[0], '  [downloading]' )
            osm_data = ox.geometries_from_place( fields[0], tags )
            if not len(osm_data)==0:
                fname = dir_osmdata + "buildings__" + fields[0] + ".geojson"
                osm_data.to_file( fname, driver='GeoJSON' )
            else:
                print("%45s" % "Skipped because it's empty.")
        except OSError as err:
            print("  OS error:", err)
        except ValueError:
            print("%45s" % "Could not find.")
        except Exception as err:
            print(f"  Unexpected {err=}, {type(err)=}")
            raise

#### OSM data :: MANUAL psql insert

In [ ]:
dir_osmdata = "osm_data/"
tag = "buildings"
CRS = 4326
geom_used = ["Polygon", "MultiPolygon"]

In [ ]:
filename = 'buildings__Afragola.geojson'

In [ ]:
city = filename.split("__")[1].split(".geo")[0]
city

In [ ]:
#tmp = geopandas.read_file("osm_data/buildings__Lettere.geojson")
tmp = geopandas.read_file( dir_osmdata + filename )

In [ ]:
tmp.head(1)

In [ ]:
tmp[tmp["addr:city"]==city].shape

In [ ]:
tmp["addr:city"]=city

In [ ]:
tmp[tmp["addr:city"]==city].shape

In [ ]:
geom_types = tmp.geometry.type.unique()
for g in geom_types:
    nrows = tmp[tmp.geometry.type==g].shape[0]
    if g in geom_used:
        print( "  > " + g + " [included %d]" % nrows )
    else:
        print( "  > " + g + " [discarded] %d]" % nrows )
        tmp = tmp.drop( tmp[tmp.geometry.type==g].index )

In [ ]:
from impervious.config import pg_url  # connessione da .env
db_connection_url = pg_url()
con = create_engine(db_connection_url)

In [ ]:
# test connection
inspector = inspect(con)
print( inspector.get_schema_names() )
print( inspector.get_table_names( 'public' ) )

In [ ]:
tbl_names = inspector.get_table_names( 'public' )
tbl_names

In [ ]:
if tag in tbl_names:
    print('Found table *%s* in db:OSM schema:public' % tag)

In [ ]:
tmp.to_postgis("buildings", con, schema="public", if_exists='append', index=True) #, dtype={'geometry': Geometry('POLYGON', srid=4326)})

#### OSM data :: BATCH psql insert

##### fix issues before I could run the bathc procedure below

ISSUE #01 :: geojson files have different sets of columns

In [ ]:
#contact:housenumber

In [ ]:
a = geopandas.read_file("osm_data/buildings__Angri.geojson")
b = geopandas.read_file("osm_data/buildings__Afragola.geojson")

In [ ]:
a.head(1)

In [ ]:
b.head(1)

In [ ]:
a.shape

In [ ]:
b.shape

In [ ]:
ac = list(a.columns)
bc = list(b.columns)

In [ ]:
ac

In [ ]:
bc

In [ ]:
# setxor:
set(ac).symmetric_difference(bc)

**Final remark:**
I decide to keep only few columns to avoid wasting time.

In [ ]:
column_used = ['element_type','osmid','nodes','building','addr:city','geometry']

In [ ]:
a[column_used].shape

In [ ]:
b[column_used].shape

In [ ]:
c = b.drop(columns='osmid')

In [ ]:
c[column_used].shape

In [ ]:
c.shape

In [ ]:
list( set(list(c.columns)) & set(column_used) )

In [ ]:
# change a | b | c below in *.columns:
if len(list( set(list(c.columns)) & set(column_used) )) < len(column_used):
    print('error')
else:
    print('good')

In [ ]:
# columns that are missing in c.columns compared to column_used:
not_found = list(set(column_used) - set(c.columns))
not_found

In [ ]:
c.insert(loc=len(c.columns),column='osmid',value=None)

In [ ]:
# columns that are missing in c.columns compared to column_used:
not_found = list(set(column_used) - set(c.columns))
not_found

In [ ]:
c = b.drop(columns=['osmid','nodes'])
not_found = list(set(column_used) - set(c.columns))
print(not_found)
for nf in not_found:
    c.insert(loc=len(c.columns)-1,column=nf,value=None)

not_found = list(set(column_used) - set(c.columns))
print(not_found)

##### Run BATCH

Database pgis-osm installed in docker on Pedometrics-VM

In [ ]:
dir_osmdata = "osm_data/"
tag = "buildings"
CRS = 4326
geom_used = ["Polygon", "MultiPolygon"]
column_used = ['element_type','osmid','nodes','building','addr:city','geometry']

In [ ]:
from impervious.config import pg_url  # connessione da .env
row=0
tmp = filename = None

try:
    db_connection_url = pg_url()
    con = create_engine(db_connection_url)
    # test connection
    inspector = inspect(con)
    tbl_names = inspector.get_table_names( 'public' )
    if tag in tbl_names:
        print('Found table *%s* in db:OSM schema:public' % tag)
    else:
        print('Table *%s* not found in db:OSM schema:public' % tag)
except OSError as err:
    print("  OS error:", err)
except ValueError:
    print("%45s" % "Could not connect to postGIS")
except Exception as err:
    print(f"  Unexpected {err=}, {type(err)=}")
    raise


for filename in os.listdir( dir_osmdata ):
    if filename.startswith( tag + "__" ):
        row = row +1
        try:
            print( "%04d" % row, "%40s" % filename, ' ...' )

            # step #00 :: Read geojson
            tmp = geopandas.read_file( dir_osmdata + filename )
            
            # step #01 :: select column names
            not_found = list(set(column_used) - set(tmp.columns))
            for nf in not_found:
                print( "  > add missing column %s" % nf )
                tmp.insert(loc=len(tmp.columns)-1,column=nf,value=None)

            not_found = list(set(column_used) - set(c.columns))
            if not len(not_found)==0:
                print('Tag: %s, City: %s :: skipped for issues on missing columns')
                continue
            
            tmp = tmp[column_used]
            
            # step #02 :: write city name, unfortunately it is often/always missing
            city = filename.split("__")[1].split(".geo")[0]
            tmp["addr:city"]=city
            
            # step 03 :: discard geometries not included in 'geom_used'
            geom_types = tmp.geometry.type.unique()
            for g in geom_types:
                nrows = tmp[tmp.geometry.type==g].shape[0]
                if g in geom_used:
                    print( "  > " + g + " [included %d]" % nrows )
                else:
                    print( "  > " + g + " [discarded] %d]" % nrows )
                    tmp = tmp.drop( tmp[tmp.geometry.type==g].index )

            # step 04 :: write in PostGIS
            tmp.to_postgis(tag, con, schema="public", if_exists='append', index=True)
            
            tmp = None
            print("")

        except OSError as err:
            print("  OS error:", err)
        except ValueError:
            print("%45s" % "Could not PSQL insert")
        except Exception as err:
            print(f"  Unexpected {err=}, {type(err)=}")
            raise

#### Reproject OSM data

Read geojson

In [ ]:
tmp = geopandas.read_file("osm_data/buildings__Lettere.geojson")

Reproject

In [ ]:
tmp = tmp.to_crs(32633)

Remove possible Point geometries

In [ ]:
tmp.geom_type.unique()

Save geojson (overwrite)